# Conformational Soft Hash (CSH) — full analysis

Single notebook covering everything reported in the manuscript. Requires
`csh_hash.py` (v2) in the same directory.

Pipeline: canonical atom order -> distance and torsion features -> scaling +
PCA fitted on a reference set of conformers of the same molecule -> 256-bit
SimHash. Nothing is fitted on the conformers being evaluated.

| section | produces | runtime |
|---|---|---|
| 1 | sanity checks (round-trip, determinism, set independence) | seconds |
| 2 | molecule sets, SMILES validated by molecular weight | seconds |
| 3 | justification for DM-RMSD (alignment artefact) | ~1 min |
| 4 | benchmark function | — |
| 5 | development set, 28 molecules -> Table 1 | ~15 min |
| 6 | control set, 12 molecules -> Table 2 | ~5 min |
| 7 | summary statistics, paired tests | seconds |
| 8 | hash length and PCA dimension study | ~10 min |
| 9 | figures | seconds |
| 10 | reference-ensemble overlap check | ~3 min |
| 11 | blockchain registration — RUN LAST, writes are irreversible | minutes |

Sections 1-10 can be re-run freely. Section 11 writes to a public blockchain
and cannot be undone; run it only when the hashes are final.

In [3]:
# ── Imports and configuration ────────────────────────────────────────────────
import csv, json, time, os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, wilcoxon
from scipy.spatial.distance import pdist

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdMolAlign
from rdkit.Chem import TorsionFingerprints as TF
from rdkit.Chem.rdMolDescriptors import GetUSR, GetUSRCAT, GetUSRScore
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
RDLogger.DisableLog("rdApp.*")

from csh_hash import CSHHasher, raw_feature_vector, REF_SEED, PROJECTION_SEED, REF_N_CONFS

N_CONFS    = 50     # conformers per molecule for evaluation
TEST_SEED  = 42     # evaluation ensemble; MUST differ from REF_SEED
N_BITS     = 256    # hash length; one 32-byte Ethereum storage slot
PCA_DIMS   = 32
HEAVY      = True   # heavy-atom variant: all methods see the same atoms
QUICK      = False  # True -> skip paclitaxel and cyclosporin A

print(f"RDKit {Chem.rdBase.rdkitVersion}")
print(f"fit seed {REF_SEED} | test seed {TEST_SEED} | {N_BITS}-bit hash | PCA {PCA_DIMS}")

RDKit 2026.03.1
fit seed 7 | test seed 42 | 256-bit hash | PCA 32


## 1. Sanity checks

Three properties the registry depends on. All must hold before any benchmark
number means anything.

1. **Round-trip** — the same geometry submitted with a different atom order
   must give the same hash.
2. **Determinism** — a transform rebuilt from scratch must reproduce the hash,
   i.e. two independent users get identical results.
3. **Set independence** — the hash of one conformer must not depend on which
   other conformers happen to be in the file. This is what the original
   pipeline violated.

In [4]:
# ── Sanity checks ────────────────────────────────────────────────────────────
def embed(smiles, n, seed=TEST_SEED):
    m = Chem.AddHs(Chem.MolFromSmiles(smiles))
    p = AllChem.ETKDGv3(); p.randomSeed = seed
    AllChem.EmbedMultipleConfs(m, numConfs=n, params=p)
    AllChem.MMFFOptimizeMoleculeConfs(m)
    return m

CHECKS = [("Aspirin",  "CC(=O)Oc1ccccc1C(=O)O"),
          ("Caffeine", "Cn1cnc2c1c(=O)n(C)c(=O)n2C"),
          ("Imatinib", "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1")]

for name, smi in CHECKS:
    H = CSHHasher(smi, heavy=HEAVY, pca_dims=PCA_DIMS, n_bits=N_BITS)
    m = embed(smi, 5)
    ref = H.hash_bits(m, 0)

    rng = np.random.RandomState(0)
    roundtrip = all(
        np.array_equal(H.hash_bits(Chem.RenumberAtoms(
            m, [int(x) for x in rng.permutation(m.GetNumAtoms())]), 0), ref)
        for _ in range(8))

    # Clear the reference-ensemble cache first: without this, the second
    # hasher would reuse the first one's ensemble and the test would pass
    # trivially instead of checking that the ensemble itself is reproducible.
    CSHHasher.clear_reference_cache()
    determinism = (CSHHasher(smi, heavy=HEAVY, pca_dims=PCA_DIMS, n_bits=N_BITS).hash_hex(m, 0)
                   == H.hash_hex(m, 0))
    alone = embed(smi, 1)
    set_independence = (H.hash_hex(alone, 0) == H.hash_hex(m, 0))

    ok = lambda b: "PASS" if b else "FAIL"
    print(f"{name:<10} round-trip {ok(roundtrip)} | determinism {ok(determinism)} | "
          f"set-independence {ok(set_independence)}")

Aspirin    round-trip PASS | determinism PASS | set-independence PASS
Caffeine   round-trip PASS | determinism PASS | set-independence PASS
Imatinib   round-trip PASS | determinism PASS | set-independence PASS


## 2. Molecule sets

**Development set** (28) — the molecules used while developing the method.
Ten of the SMILES strings in the original version did not correspond to the
compounds they were named after; all structures here are taken from PubChem.

**Control set** (12) — molecules selected after the method was finalised, so
nothing could be tuned to them.

Every SMILES is checked against a reference molecular weight below. A mismatch
means the string is not the compound it claims to be.

In [5]:
# ── Molecule sets (name, SMILES, reference MW) ───────────────────────────────
DEV = [
 ("Aspirin","CC(=O)Oc1ccccc1C(=O)O",180.16),
 ("Paracetamol","CC(=O)Nc1ccc(O)cc1",151.16),
 ("Caffeine","Cn1cnc2c1c(=O)n(C)c(=O)n2C",194.19),
 ("Phenol","Oc1ccccc1",94.11),
 ("Aniline","Nc1ccccc1",93.13),
 ("Ibuprofen","CC(C)Cc1ccc(cc1)C(C)C(=O)O",206.28),
 ("Diazepam","CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21",284.74),
 ("Naproxen","CC(C(=O)O)c1ccc2cccc(OC)c2c1",230.26),
 ("Ketoprofen","CC(C(=O)O)c1cccc(c1)C(=O)c1ccccc1",254.28),
 ("Lorazepam","OC1N=C(c2ccccc2Cl)c2cc(Cl)ccc2NC1=O",321.16),
 ("Warfarin","CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O",308.33),
 ("Metformin","CN(C)C(=N)NC(=N)N",129.16),
 ("Lidocaine","CCN(CC)CC(=O)Nc1c(C)cccc1C",234.34),
 ("Propranolol","CC(C)NCC(O)COc1cccc2ccccc12",259.34),
 ("Atenolol","CC(C)NCC(O)COc1ccc(CC(N)=O)cc1",266.34),
 ("Tamoxifen","CCC(=C(c1ccccc1)c1ccc(OCCN(C)C)cc1)c1ccccc1",371.51),
 ("Imatinib","Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",493.60),
 ("Amlodipine","CCOC(=O)C1=C(COCCN)NC(C)=C(C1c1ccccc1Cl)C(=O)OC",408.88),
 ("Sildenafil","CCCc1nn(C)c2c(=O)[nH]c(-c3cc(S(=O)(=O)N4CCN(C)CC4)ccc3OCC)nc12",474.58),
 ("Valsartan","CCCCC(=O)N(Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1)C(C(C)C)C(=O)O",435.52),
 ("Losartan","CCCCc1nc(Cl)c(CO)n1Cc1ccc(-c2ccccc2-c2nnn[nH]2)cc1",422.91),
 ("Celecoxib","Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1",381.37),
 ("Atorvastatin","CC(C)c1c(C(=O)Nc2ccccc2)c(-c2ccccc2)c(-c2ccc(F)cc2)n1CC[C@@H](O)C[C@@H](O)CC(=O)O",558.64),
 ("Paclitaxel","CC(=O)O[C@@H]1C(=O)[C@]2(C)[C@@H](O)C[C@H]3OC[C@@]3(OC(C)=O)[C@H]2[C@H](OC(=O)c2ccccc2)[C@]2(O)C[C@H](OC(=O)[C@H](O)[C@@H](NC(=O)c3ccccc3)c3ccccc3)C(C)=C1C2(C)C",853.91),
 ("Cyclosporin_A","CC[C@H]1NC(=O)[C@@H]([C@H](O)[C@H](C)C/C=C/C)N(C)C(=O)[C@H](C(C)C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H](C)NC(=O)[C@H](C)NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H](C(C)C)NC(=O)[C@H](CC(C)C)N(C)C(=O)CN(C)C1=O",1202.61),
 ("Haloperidol","O=C(CCCN1CCC(O)(c2ccc(Cl)cc2)CC1)c1ccc(F)cc1",375.86),
 ("Risperidone","CC1=C(CCN2CCC(c3noc4cc(F)ccc34)CC2)C(=O)N2CCCCC2=N1",410.48),
 ("Ciprofloxacin","O=C(O)c1cn(C2CC2)c2cc(N3CCNCC3)c(F)cc2c1=O",331.34),
]

CONTROL = [
 ("Ketamine","CNC1(c2ccccc2Cl)CCCCC1=O",237.72),
 ("Fluoxetine","CNCCC(Oc1ccc(C(F)(F)F)cc1)c1ccccc1",309.33),
 ("Sertraline","CNC1CCC(c2ccc(Cl)c(Cl)c2)c2ccccc21",306.23),
 ("Omeprazole","COc1ccc2[nH]c(S(=O)Cc3ncc(C)c(OC)c3C)nc2c1",345.42),
 ("Diclofenac","OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl",296.15),
 ("Metoprolol","COCCc1ccc(OCC(O)CNC(C)C)cc1",267.36),
 ("Loratadine","CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1",382.88),
 ("Verapamil","COc1ccc(CCN(C)CCCC(C#N)(C(C)C)c2ccc(OC)c(OC)c2)cc1OC",454.61),
 ("Simvastatin","CCC(C)(C)C(=O)O[C@H]1C[C@H](C)C=C2C=C[C@H](C)[C@H](CC[C@@H]3C[C@H](O)CC(=O)O3)[C@@H]21",418.57),
 ("Gefitinib","COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",446.90),
 ("Amitriptyline","CN(C)CCC=C1c2ccccc2CCc2ccccc21",277.41),
 ("Nifedipine","COC(=O)C1=C(C)NC(C)=C(C(=O)OC)C1c1ccccc1[N+](=O)[O-]",346.33),
]

def validate(label, molecules, tol=1.0):
    bad = []
    for name, smi, ref in molecules:
        m = Chem.MolFromSmiles(smi)
        if m is None:
            bad.append((name, "INVALID SMILES")); continue
        mw = Descriptors.MolWt(m)
        if abs(mw - ref) > tol:
            bad.append((name, f"MW {mw:.2f} vs reference {ref:.2f}"))
    print(f"{label}: {len(molecules)} molecules, {len(bad)} problems" +
          ("" if not bad else " -> " + "; ".join(f"{n}: {w}" for n, w in bad)))
    return not bad

validate("Development set", DEV)
validate("Control set    ", CONTROL)

Development set: 28 molecules, 0 problems
Control set    : 12 molecules, 0 problems


True

## 3. Two reference metrics, and where they disagree

Two rotation-invariant distances between conformers are available, and they
measure different things:

- **Aligned RMSD** (`GetBestRMS`) superimposes the two conformers and allows
  atoms to map onto symmetry-equivalent positions. It answers: are these the
  same shape, ignoring which equivalent atom is which?
- **DM-RMSD** compares the two distance matrices at fixed atom indices. It
  answers: are the interatomic distances the same, for these specific atoms?

Where a molecule has symmetry, the two can disagree: a pair may be equivalent
under `GetBestRMS` while DM-RMSD sees a difference, because DM-RMSD does not
permute atoms. Which value is "correct" depends on the question being asked,
and this notebook does not adjudicate it.

The cell below counts such pairs. It is reported as a property of the two
metrics, not as evidence that either is wrong. DM-RMSD is used as the primary
reference throughout because CSH operates on fixed canonical atom indices, and
TFD is reported alongside as an independent check on a quantity that is not
built from interatomic distances at all.

In [6]:
# ── Where the two reference metrics disagree ─────────────────────────────────
def dm_rmsd(mol, i, j, heavy=HEAVY):
    """Rotation-invariant distance between two conformers of the same molecule."""
    w = Chem.RemoveHs(mol) if heavy else mol
    a = pdist(w.GetConformer(i).GetPositions())
    b = pdist(w.GetConformer(j).GetPositions())
    return float(np.sqrt(np.mean((a - b) ** 2)))

demo = embed("CC(C)Cc1ccc(cc1)C(C)C(=O)O", 30)      # ibuprofen
nc = demo.GetNumConformers()
rows, agree = [], 0
for i in range(nc):
    for j in range(i + 1, nc):
        r = rdMolAlign.GetBestRMS(demo, demo, i, j)
        if r < 0.01:
            d = dm_rmsd(demo, i, j)
            if d < 0.01:
                agree += 1                 # both metrics call the pair identical
            else:
                rows.append((i, j, r, d))

print(f"pairs with aligned RMSD < 0.01 A: {len(rows) + agree}")
print(f"  both metrics agree the pair is identical: {agree}")
print(f"  DM-RMSD reports a difference:             {len(rows)}\n")
print("The second group is where symmetry-equivalent atom mappings matter.")
print("Neither metric is taken as ground truth here.\n")
for i, j, r, d in rows[:5]:
    print(f"  conf {i:2d} vs {j:2d}: aligned RMSD = {r:.6f} A, DM-RMSD = {d:.4f} A")
if not rows:
    print("  the two metrics agree on every such pair in this ensemble")

pairs with aligned RMSD < 0.01 A: 22
  both metrics agree the pair is identical: 4
  DM-RMSD reports a difference:             18

The second group is where symmetry-equivalent atom mappings matter.
Neither metric is taken as ground truth here.

  conf  1 vs 21: aligned RMSD = 0.000031 A, DM-RMSD = 0.5652 A
  conf  1 vs 29: aligned RMSD = 0.000026 A, DM-RMSD = 0.5652 A
  conf  2 vs  6: aligned RMSD = 0.000005 A, DM-RMSD = 0.6025 A
  conf  2 vs 22: aligned RMSD = 0.000031 A, DM-RMSD = 0.5644 A
  conf  3 vs  4: aligned RMSD = 0.000019 A, DM-RMSD = 0.5656 A


## 4. Benchmark

For each molecule: generate `N_CONFS` conformers, then correlate the reference
metric against the similarity reported by each method, over all conformer pairs.
Every method sees the same atoms.

The methods differ in how they reach a fixed-size representation, and that
difference is a property of each method rather than a choice made here:

- **CSH** — feature width varies with the molecule (number of atom pairs), so a
  per-molecule scaling and PCA are required. Hashes of different molecules do
  not share a frame of reference.
- **USR, USRCAT** — fixed width (12 and 60 values), so one global projection
  serves every molecule; only centring statistics are needed, computed once.
- **3DAPfp, 3DXfp** — built from the same interatomic distances as CSH, but
  binned onto a fixed 16-point grid, which gives a constant width (16 and 80
  values) without any per-molecule fitting. The closest published relative of
  CSH.
- **E3FP** — natively binary at 256 bits, so no quantisation step exists;
  compared by Tanimoto, its own measure.

As a control, USR and USRCAT are additionally forced through the per-molecule
CSH pipeline, to show whether differences come from the descriptors or from
unequal preprocessing.

In [7]:
# ── Additional baselines: 3D atom-pair fingerprints and E3FP ─────────────────
# 3DAPfp (Awale, Jin & Reymond, J Cheminform 2015;7:3) bins through-space
# atom-pair distances onto a fixed 16-point grid. Like CSH it is built from
# interatomic distances, but the fixed grid gives it a constant width, so it
# needs no per-molecule transform and stays comparable across molecules. It is
# therefore the closest published relative of CSH and the most demanding
# baseline for it. The 80-value 3DXfp variant is not included: it requires
# pharmacophore typing that we cannot reimplement faithfully.
from ap3d import apfp3d

# E3FP (Axen et al., J Med Chem 2017;60:7393) is natively binary and encodes
# local 3D atom environments. It needs no quantisation step, so it enters the
# comparison at 256 bits by construction and is compared by Tanimoto.
import logging
logging.disable(logging.WARNING)
try:
    from e3fp.fingerprint.generate import fprints_dict_from_mol
    E3FP_AVAILABLE = True
except ImportError:
    E3FP_AVAILABLE = False
    print("e3fp not installed (pip install e3fp) — E3FP columns will be nan")

def e3fp_bits(mol, name, bits=N_BITS, level=5):
    """E3FP folded to `bits` for every conformer, as a list of uint8 arrays.

    E3FP is natively binary but is normally used at 1024 bits or more; folding
    to 256 introduces collisions, so this is a folded variant and should be
    described as such.
    """
    if not E3FP_AVAILABLE:
        return None
    try:
        m = Chem.RemoveHs(Chem.Mol(mol)) if HEAVY else Chem.Mol(mol)
        m.SetProp("_Name", str(name))
        d = fprints_dict_from_mol(m, bits=bits, level=level, first=-1)
        fps = d[max(d.keys())]
        return [f.to_vector(sparse=False).astype(np.uint8) for f in fps]
    except Exception:
        return None

print(f"3DAPfp/3DXfp: ready | E3FP: {'ready' if E3FP_AVAILABLE else 'unavailable'}")

3DAPfp/3DXfp: ready | E3FP: ready


In [8]:
# ── Global baseline statistics ───────────────────────────────────────────────
# USR and USRCAT vectors are moments of distance distributions: all positive,
# with a large constant offset. Projected raw, every conformer lands on almost
# the same side of every hyperplane and the hash collapses. They therefore need
# centring — but because their width is fixed, the statistics can be computed
# ONCE, pooled over many molecules, and published as constants. No reference
# ensemble per molecule, and hashes stay comparable across molecules.

GLOBAL_STATS_MOLECULES = [n for n, _, _ in DEV]      # pooled reference ensembles

def compute_global_stats(names, n=20):
    smiles_by_name = {n_: s for n_, s, _ in DEV}
    U, C, A = [], [], []
    for name in names:
        ref = embed(smiles_by_name[name], n, seed=REF_SEED)
        w = Chem.RemoveHs(ref) if HEAVY else ref
        for c in range(ref.GetNumConformers()):
            U.append(list(GetUSR(w, c)))
            C.append(list(GetUSRCAT(w, None, c)))
            A.append(apfp3d(ref, c, heavy=HEAVY))
    U, C, A = np.array(U), np.array(C), np.array(A)
    return ((U.mean(0), U.std(0) + 1e-9),
            (C.mean(0), C.std(0) + 1e-9),
            (A.mean(0), A.std(0) + 1e-9))

print("computing global baseline statistics (once, pooled over the dev set)...")
(USR_MEAN, USR_STD), (CAT_MEAN, CAT_STD), (AP_MEAN, AP_STD) = \
    compute_global_stats(GLOBAL_STATS_MOLECULES)
print(f"USR:    {len(USR_MEAN)} dimensions")
print(f"USRCAT: {len(CAT_MEAN)} dimensions")
print(f"3DAPfp: {len(AP_MEAN)} dimensions")
print("These are single global constants — the same for every molecule.")

computing global baseline statistics (once, pooled over the dev set)...


KeyboardInterrupt: Embedding cancelled

In [ ]:
# ── Benchmark ────────────────────────────────────────────────────────────────
def simhash(V, n_bits=N_BITS, seed=PROJECTION_SEED):
    """Project onto n_bits random hyperplanes. Input width is arbitrary."""
    rng = np.random.RandomState(seed)
    return (V @ rng.randn(n_bits, V.shape[1]).T > 0).astype(np.uint8)

def usrcat_similarity(a, b):
    """USRCAT similarity; GetUSRScore only accepts the 12-value USR vector."""
    return 1.0 / (1.0 + np.mean(np.abs(np.asarray(a) - np.asarray(b))))

def reference_transform(smiles, kind, n=50, npca=PCA_DIMS):
    """Scaling + PCA fitted per molecule — the control condition only."""
    ref = embed(smiles, n, seed=REF_SEED)
    w = Chem.RemoveHs(ref) if HEAVY else ref
    X = np.array([list(GetUSR(w, c)) if kind == "usr" else list(GetUSRCAT(w, None, c))
                  for c in range(ref.GetNumConformers())])
    sc = StandardScaler().fit(X)
    k = min(npca, X.shape[0] - 1, X.shape[1])
    return sc, PCA(n_components=k, random_state=PROJECTION_SEED).fit(sc.transform(X))

def benchmark_molecule(name, smiles):
    mol = embed(smiles, N_CONFS)
    nc = mol.GetNumConformers()
    if nc < 5:
        return None
    w = Chem.RemoveHs(mol) if HEAVY else mol
    pairs = [(i, j) for i in range(nc) for j in range(i + 1, nc)]

    dvec = [pdist(w.GetConformer(c).GetPositions()) for c in range(nc)]
    dm = np.array([np.sqrt(np.mean((dvec[i] - dvec[j]) ** 2)) for i, j in pairs])

    # Second reference metric: torsion fingerprint deviation. DM-RMSD is built
    # from interatomic distances, which is also what the CSH feature vector is
    # built from; TFD is built from torsion angles instead, so it does not share
    # that construction. Rigid molecules have no comparable torsions and give a
    # degenerate TFD, reported as nan.
    try:
        tfd_m = np.array(TF.GetTFDMatrix(mol))
        tfd = np.array([tfd_m[j * (j - 1) // 2 + i] for i, j in pairs])
        if np.std(tfd) == 0:
            tfd = None
    except Exception:
        tfd = None

    # --- CSH: variable feature width, so the transform is per molecule ---
    H = CSHHasher(smiles, heavy=HEAVY, pca_dims=PCA_DIMS, n_bits=N_BITS)
    F = np.array([H.feature_vector(mol, c) for c in range(nc)])
    B_csh = np.array([H.hash_bits(mol, c) for c in range(nc)])   # via the module

    usr  = np.array([list(GetUSR(w, c)) for c in range(nc)])
    ucat = np.array([list(GetUSRCAT(w, None, c)) for c in range(nc)])

    # --- baselines, global quantisation: fixed width, so one set of global
    #     constants and one shared projection serve every molecule ---
    B_usr_direct = simhash((usr  - USR_MEAN) / USR_STD)
    B_cat_direct = simhash((ucat - CAT_MEAN) / CAT_STD)

    # --- baselines, control: forced through the per-molecule CSH pipeline ---
    sc_u, pca_u = reference_transform(smiles, "usr")
    sc_c, pca_c = reference_transform(smiles, "usrcat")
    B_usr_pipe = simhash(pca_u.transform(sc_u.transform(usr)))
    B_cat_pipe = simhash(pca_c.transform(sc_c.transform(ucat)))


    # --- 3DAPfp: atom-pair distances binned onto a FIXED grid, so the
    #     width is the same for every molecule and no per-molecule transform is
    #     needed. Conceptually the closest published relative of CSH. ---
    ap = np.array([apfp3d(mol, c, heavy=HEAVY) for c in range(nc)])
    B_ap = simhash((ap - AP_MEAN) / AP_STD)      # global constants, not test data
    cityblock = lambda M: [-np.abs(M[i] - M[j]).sum() for i, j in pairs]

    # --- E3FP: natively binary at 256 bits, compared by Tanimoto, its own
    #     measure. No quantisation step at all. ---
    e3 = e3fp_bits(mol, smiles)
    if e3 is not None and len(e3) == nc:      # guard against partial generation
        tani = []
        for i, j in pairs:
            a, b = e3[i], e3[j]
            u = (a | b).sum()
            tani.append((a & b).sum() / u if u else 0.0)
    else:
        tani = None

    def rho(sim, target=None):
        """Negated Spearman between a distance and a similarity, so that higher
        is better. A sign error surfaces as a negative value instead of being
        hidden by an absolute value."""
        target = dm if target is None else target
        sim = np.asarray(sim, dtype=float)
        if target is None or np.std(sim) == 0 or np.std(target) == 0:
            return float("nan")
        return -spearmanr(target, sim).statistic
    def rho_t(sim):
        return float("nan") if tfd is None else round(rho(sim, tfd), 3)
    ham = lambda B: [np.mean(B[i] == B[j]) for i, j in pairs]
    cos = lambda X: (lambda Xn: [Xn[i] @ Xn[j] for i, j in pairs])(
        X / np.linalg.norm(X, axis=1, keepdims=True))
    Us = (usr  - USR_MEAN) / USR_STD          # globally standardised, pre-binarisation
    Cs = (ucat - CAT_MEAN) / CAT_STD

    return {"molecule": name, "n_atoms": w.GetNumAtoms(), "n_confs": nc,
            "n_feat_csh": F.shape[1],
            # spread of the reference metric: near zero means the heavy-atom
            # conformers barely differ, so any correlation is noise
            "dm_rmsd_max": round(float(np.max(dm)), 3),
            # native metric of each method
            "csh_eucl":   round(rho([-np.linalg.norm(F[i] - F[j]) for i, j in pairs]), 3),
            "usr_native": round(rho([GetUSRScore(list(usr[i]), list(usr[j])) for i, j in pairs]), 3),
            "cat_native": round(rho([usrcat_similarity(ucat[i], ucat[j]) for i, j in pairs]), 3),
            # exact cosine after the same preprocessing that precedes binarisation
            "csh_cos":    round(rho(cos(F)), 3),
            "usr_cos":    round(rho(cos(Us)), 3),
            "cat_cos":    round(rho(cos(Cs)), 3),
            "csh":        round(rho(ham(B_csh)), 3),
            "usr_bits":   round(rho(ham(B_usr_direct)), 3),
            "cat_bits":   round(rho(ham(B_cat_direct)), 3),
            "usr_pipe":   round(rho(ham(B_usr_pipe)), 3),
            "cat_pipe":   round(rho(ham(B_cat_pipe)), 3),
            # 3D atom-pair fingerprints (fixed width, no per-molecule transform)
            "ap_native":  round(rho(cityblock(ap)), 3),
            "ap_bits":    round(rho(ham(B_ap)), 3),
            # E3FP: natively binary, Tanimoto
            "e3fp":       round(rho(tani), 3) if tani is not None else float("nan"),
            # same hashes, correlated against TFD instead of DM-RMSD
            "csh_tfd":    rho_t(ham(B_csh)),
            "usr_tfd":    rho_t(ham(B_usr_direct)),
            "cat_tfd":    rho_t(ham(B_cat_direct)),
            "unique_hashes": len(set(tuple(b) for b in B_csh))}

def run_set(molecules, label, csv_path):
    rows = []
    print(f"--- {label} ({N_BITS} bits, {'heavy-atom' if HEAVY else 'all-atom'}) ---")
    for name, smiles, _ in molecules:
        t0 = time.time()
        r = benchmark_molecule(name, smiles)
        if r is None:
            print(f"  {name:<15} skipped"); continue
        rows.append(r)
        print(f"  {name:<15} CSH={r['csh']:.3f} | USR={r['usr_bits']:.3f} "
              f"| USRCAT={r['cat_bits']:.3f} | 3DAPfp={r['ap_bits']:.3f} "
              f"| E3FP={r['e3fp']:.3f}  ({time.time()-t0:.0f}s)")
    df = pd.DataFrame(rows); df.to_csv(csv_path, index=False)
    print(f"saved -> {csv_path}")
    return df

## 5. Development set — Table 1

Set `QUICK = True` in the configuration cell to skip paclitaxel and
cyclosporin A, which together account for most of the runtime.

In [ ]:
# ── Run: development set ─────────────────────────────────────────────────────
dev_molecules = [m for m in DEV
                 if not (QUICK and m[0] in ("Paclitaxel", "Cyclosporin_A"))]
df_dev = run_set(dev_molecules, "Development set", "table1_development.csv")

## 6. Validation set — Table 2

Twelve molecules chosen after the method was finalised. They were not used to
fit any parameter, but several benchmark variants were examined on them during
this work, so they are described as a validation set rather than an untouched
hold-out.

In [ ]:
# ── Run: control set ─────────────────────────────────────────────────────────
df_ctl = run_set(CONTROL, "Control set", "table2_control.csv")

## 7. Summary statistics

Each method is reported three ways, so that two separate effects can be told
apart:

- **native metric** — the similarity measure the method was defined with
  (Euclidean distance in PCA space for CSH; the USR and USRCAT scores for the
  baselines);
- **exact cosine** — cosine similarity after exactly the preprocessing that
  precedes binarisation, with no quantisation;
- **256-bit Hamming** — the hash.

SimHash approximates the *angle* between vectors, not the native metric, so the
drop from native to exact cosine is a change of metric, while the drop from
exact cosine to the hash is the true cost of binarisation. Reporting only the
difference between the native metric and the hash would conflate the two.

Differences between methods are tested pairwise, since the same molecules are
used throughout.

In [ ]:
# ── Summary statistics ───────────────────────────────────────────────────────
def bootstrap_ci(x, n_boot=20000, alpha=0.05, seed=0):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    if len(x) < 2:
        return (np.nan, np.nan)
    rng = np.random.RandomState(seed)
    means = [rng.choice(x, len(x), replace=True).mean() for _ in range(n_boot)]
    return tuple(np.round(np.percentile(means, [100*alpha/2, 100*(1-alpha/2)]), 3))

# native metric | exact cosine after preprocessing | 256-bit Hamming
ROWS = [("CSH",    "csh_eucl",   "csh_cos", "csh"),
        ("USR",    "usr_native", "usr_cos", "usr_bits"),
        ("USRCAT", "cat_native", "cat_cos", "cat_bits"),
        ("3DAPfp", "ap_native",  None,      "ap_bits"),
        ("E3FP",   "e3fp",       None,      "e3fp")]

def summarise(df, label):
    print(f"\n{'=' * 82}\n{label}  (n = {len(df)}, {N_BITS} bits)\n{'=' * 82}")
    print(f"{'method':<8} {'native':>9} {'exact cosine':>14} {'256-bit':>10} "
          f"{'metric change':>15} {'binarisation':>14}")
    print("-" * 82)
    # A method that assigns every conformer the same hash conveys no ordering
    # information, so Spearman is undefined and the cell is nan. That is a
    # result, not missing data: it is reported here rather than silently
    # dropped, because averaging over only the molecules where a method
    # happened to work flatters that method.
    n_tot = len(df)
    print(f"{'method':<8}{'collapsed':>11}{'scored':>8}   (collapsed = one hash for every conformer)")
    print("-" * 82)
    for lab, ncol, ccol, bcol in ROWS:
        n_bad = int(df[bcol].isna().sum())
        print(f"{lab:<8}{n_bad:>7}/{n_tot:<3}{n_tot - n_bad:>8}")
    rigid = df[df.dm_rmsd_max < 0.15] if "dm_rmsd_max" in df else df.iloc[:0]
    if len(rigid):
        print(f"\n{len(rigid)} molecule(s) have max DM-RMSD < 0.15 A, i.e. their heavy-atom")
        print("conformers are nearly identical. On those, a method reporting no")
        print("difference may be more faithful than one reporting a ranking:")
        print("  " + ", ".join(rigid.molecule.tolist()))
    print("-" * 82)

    for lab, ncol, ccol, bcol in ROWS:
        n_, b_ = df[ncol].mean(), df[bcol].mean()      # nan-skipping: see above
        if ccol is None:
            print(f"{lab:<8} {n_:>9.3f} {'-':>14} {b_:>10.3f} "
                  f"{'-':>15} {'-':>14}")
        else:
            c_ = df[ccol].mean()
            print(f"{lab:<8} {n_:>9.3f} {c_:>14.3f} {b_:>10.3f} "
                  f"{c_ - n_:>+15.3f} {b_ - c_:>+14.3f}")
    print("-" * 82)
    print("'metric change' = moving to cosine similarity; 'binarisation' = cost of")
    print("going from exact cosine to a 256-bit hash. Separating the two matters")
    print("because SimHash approximates the angle, not the native metric.")
    print("-" * 82)
    print("Control — baselines forced through the per-molecule CSH pipeline:")
    print(f"  USR    {df.usr_pipe.mean():.3f}   (global quantisation: {df.usr_bits.mean():.3f})")
    print(f"  USRCAT {df.cat_pipe.mean():.3f}   (global quantisation: {df.cat_bits.mean():.3f})")
    print("-" * 82)
    print(f"{'method':<8} {'mean±SD':>14} {'median':>8} {'95% CI':>18}   (256 bits)")
    for lab, _, _, bcol in ROWS:
        v = df[bcol]; lo, hi = bootstrap_ci(v)
        print(f"{lab:<8} {v.mean():>7.3f}±{v.std():<6.3f} {v.median():>8.3f} "
              f"{f'[{lo:.3f}, {hi:.3f}]':>18}")
    print("-" * 82)
    t = df[["csh_tfd", "usr_tfd", "cat_tfd"]].dropna()
    if len(t):
        print(f"Second reference metric (TFD, torsion-based; n = {len(t)} of {len(df)}):")
        print(f"  CSH {t.csh_tfd.mean():.3f} | USR {t.usr_tfd.mean():.3f} "
              f"| USRCAT {t.cat_tfd.mean():.3f}")
        print("  DM-RMSD is built from interatomic distances, as is the CSH feature")
        print("  vector; TFD is built from torsions and does not share that construction.")
        print("-" * 82)
    for lab, bcol in [("USR", "usr_bits"), ("USRCAT", "cat_bits"),
                      ("3DAPfp", "ap_bits"), ("E3FP", "e3fp")]:
        # paired on molecules where BOTH methods return a value, so the two
        # means are computed over the same molecules
        d = (df.csh - df[bcol]).dropna()
        if len(d) < 2:
            print(f"  CSH - {lab:<8} too few molecules scored by both methods")
            continue
        lo, hi = bootstrap_ci(d)
        p = wilcoxon(df.csh[d.index], df[bcol][d.index]).pvalue
        print(f"  CSH - {lab:<7} at 256 bits: mean {d.mean():+.3f}, median {d.median():+.3f}, "
              f"95% CI [{lo:+.3f}, {hi:+.3f}], CSH higher in {(d>0).sum()}/{len(d)}, "
              f"Wilcoxon p = {p:.3f}")

summarise(df_dev, "DEVELOPMENT SET")
summarise(df_ctl, "CONTROL SET")

## 8. Hash length and PCA dimension

Two design parameters, both reported in the Methods. The conformers and the
reference transform are computed once per molecule and reused across settings,
so this is cheaper than it looks.

In [ ]:
# ── Hash length and PCA dimension ────────────────────────────────────────────
BITS  = [64, 128, 256, 512]
# A reference ensemble of REF_N_CONFS conformers yields at most
# REF_N_CONFS - 1 components, so anything above that is silently clipped
# and would appear as a distinct setting while being identical.
PCAS  = [8, 16, 32, REF_N_CONFS - 1]
subset = [m for m in DEV if not (QUICK and m[0] in ("Paclitaxel", "Cyclosporin_A"))]

res = {(p, b): [] for p in PCAS for b in BITS}
for name, smiles, _ in subset:
    mol = embed(smiles, N_CONFS)
    nc = mol.GetNumConformers()
    w = Chem.RemoveHs(mol) if HEAVY else mol
    pairs = [(i, j) for i in range(nc) for j in range(i + 1, nc)]
    dvec = [pdist(w.GetConformer(c).GetPositions()) for c in range(nc)]
    dm = np.array([np.sqrt(np.mean((dvec[i] - dvec[j]) ** 2)) for i, j in pairs])
    # The reference ensemble is cached inside CSHHasher, so sweeping pca_dims
    # here costs one ensemble per molecule rather than one per setting.
    for npca in PCAS:
        H = CSHHasher(smiles, heavy=HEAVY, pca_dims=npca, n_bits=N_BITS)
        F = np.array([H.feature_vector(mol, c) for c in range(nc)])
        for nb in BITS:
            B = simhash(F, nb)
            sim = np.array([np.mean(B[i] == B[j]) for i, j in pairs])
            r = -spearmanr(dm, sim).statistic if np.std(sim) > 0 else np.nan
            res[(npca, nb)].append(r)
    print(f"  {name} done", flush=True)

print(f"\nMean Spearman rho, development set (n = {len(subset)})\n")
print(f"{'PCA':>5} " + "".join(f"{str(b) + ' bits':>12}" for b in BITS))
print("-" * (5 + 12 * len(BITS)))
for npca in PCAS:
    print(f"{npca:>5} " + "".join(f"{np.nanmean(res[(npca, b)]):>12.3f}" for b in BITS))

## 9. Figures

One panel per set. Molecules run down the vertical axis, ordered by maximum
DM-RMSD, so those whose heavy-atom conformers barely differ sit at the bottom
and are shaded. A method that assigned one hash to every conformer is marked
with a cross rather than drawn as a zero-height bar, since the correlation is
undefined rather than nil.

Figures are numbered 1 and 2 to match the manuscript, where the earlier
feature-space section was withdrawn.

In [ ]:
# ── Figures ──────────────────────────────────────────────────────────────────
METHODS = [("CSH",    "csh",      "#d95f02"),
           ("USR",    "usr_bits", "#1f77b4"),
           ("USRCAT", "cat_bits", "#7570b3"),
           ("3DAPfp", "ap_bits",  "#1b9e77"),
           ("E3FP",   "e3fp",     "#999999")]

FIG_NOTE = (
    "Molecules are ordered by maximum DM-RMSD, so those whose heavy-atom "
    "conformers barely differ sit at the bottom.\n"
    "A cross marks a molecule where the method assigned one hash to every "
    "conformer, leaving \u03c1 undefined.\n"
    f"All methods use a {N_BITS}-bit budget; E3FP is compared by Tanimoto, "
    "the others by normalised Hamming distance.")


def figure(df, title, path, note=FIG_NOTE, height_per_molecule=0.42):
    """Horizontal grouped bars: one group per molecule, one bar per method."""
    d = df.sort_values("dm_rmsd_max").reset_index(drop=True)
    n = len(d)
    y = np.arange(n)
    h = 0.16

    fig, ax = plt.subplots(figsize=(7.2, max(4, n * height_per_molecule) + 2.1),
                           dpi=300)
    cols = [c for _, c, _ in METHODS]
    lo = min(-0.10, float(np.nanmin(d[cols].values)) - 0.05)

    for k, (label, col, colour) in enumerate(METHODS):
        v = d[col].values.astype(float)
        off = (k - 2) * h
        ax.barh(y + off, np.nan_to_num(v), h, label=label, color=colour,
                edgecolor="none", zorder=3)
        for i, val in enumerate(v):
            if np.isnan(val):          # undefined, not zero
                ax.plot(0.012, y[i] + off, marker="x", ms=3.4,
                        color=colour, zorder=4)

    ax.set_xlim(lo, 1.05)
    ax.set_ylim(-0.6, n - 0.4)

    rigid = d.dm_rmsd_max < 0.15
    if rigid.any():
        ax.axhspan(-0.6, rigid.sum() - 0.5, color="#f4f4f4", zorder=0)
        ax.text(1.02, (rigid.sum() - 1) / 2 - 0.05, "near-rigid:\nnothing to rank",
                ha="right", va="center", fontsize=7.5, color="#999999", zorder=5)

    ax.set_yticks(y)
    ax.set_yticklabels([m.replace("_", " ") for m in d.molecule], fontsize=8.5)
    ax.set_xlabel("Spearman \u03c1 vs DM-RMSD", fontsize=9.5)
    ax.axvline(0, color="#666666", lw=0.7, zorder=2)
    ax.grid(axis="x", color="#dddddd", lw=0.6, zorder=1)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # title and legend placed on the figure, so they cannot overlap the axes
    fig.suptitle(title, fontsize=12, fontweight="bold", y=0.995)
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, ncol=len(METHODS), fontsize=8.8, frameon=False,
               loc="upper center", bbox_to_anchor=(0.5, 0.972))
    fig.text(0.015, 0.008, note, fontsize=7.3, color="#666666", va="bottom")

    plt.tight_layout(rect=[0, 0.055, 1, 0.955])
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"saved -> {path}")
    plt.show()


figure(df_dev, f"Development set ({len(df_dev)} molecules)",
       "figure1_development.png")
figure(df_ctl, f"Validation set ({len(df_ctl)} molecules)",
       "figure2_validation.png", height_per_molecule=0.52)

## 10. Reference-ensemble overlap

The transform is fitted on a reference ensemble (seed 7) and evaluated on a
separate one (seed 42). The two are generated independently, but conformer
generation converges on the same local minima, so some conformers coincide.

Excluding them is not a cleaner test — it removes the most populated minima and
leaves the rare ones. The cell measures how many coincide and what happens to
the correlation when they are dropped.

In [ ]:
# ── Reference-ensemble overlap ───────────────────────────────────────────────
EPS = 1e-4

def overlap_effect(name, smiles):
    ref, test = embed(smiles, N_CONFS, REF_SEED), embed(smiles, N_CONFS, TEST_SEED)
    wr = Chem.RemoveHs(ref)  if HEAVY else ref
    wt = Chem.RemoveHs(test) if HEAVY else test
    dr = [pdist(wr.GetConformer(c).GetPositions()) for c in range(ref.GetNumConformers())]
    dt = [pdist(wt.GetConformer(c).GetPositions()) for c in range(test.GetNumConformers())]
    shared = [i for i, b in enumerate(dt)
              if min(float(np.sqrt(np.mean((b - a) ** 2))) for a in dr) < EPS]

    H = CSHHasher(smiles, heavy=HEAVY, pca_dims=PCA_DIMS, n_bits=N_BITS)
    B = simhash(np.array([H.feature_vector(test, c)
                          for c in range(test.GetNumConformers())]))

    def rho(idx):
        pr = [(i, j) for k, i in enumerate(idx) for j in idx[k + 1:]]
        if len(pr) < 20:
            return float("nan")
        dm = np.array([np.sqrt(np.mean((dt[i] - dt[j]) ** 2)) for i, j in pr])
        sim = np.array([np.mean(B[i] == B[j]) for i, j in pr])
        return -spearmanr(dm, sim).statistic if np.std(sim) > 0 else float("nan")

    every = list(range(len(dt)))
    return {"molecule": name, "shared": f"{len(shared)}/{len(dt)}",
            "rho_all": round(rho(every), 3),
            "rho_excluding_shared": round(rho([i for i in every if i not in shared]), 3)}

ov = pd.DataFrame([overlap_effect(n, s) for n, s, _ in CONTROL[:8]])
ov["delta"] = (ov.rho_excluding_shared - ov.rho_all).round(3)
print(ov.to_string(index=False))
print(f"\nmean rho: {ov.rho_all.mean():.3f} (all) vs "
      f"{ov.rho_excluding_shared.mean():.3f} (excluding shared), "
      f"difference {ov.rho_excluding_shared.mean() - ov.rho_all.mean():+.3f}")

## 11. Blockchain registration

**Run this last.** Writes to a public chain cannot be undone, so register only
hashes that are final. `REGISTER = False` by default.

A soft hash of this kind is meaningful only within one molecule and one
transform version: two hashes from different molecules sit in different frames
of reference, and their Hamming distance means nothing. The registry makes that
scope structural. Conformers are registered into a *collection* keyed by
`(molecule, specVersion)`, and no function in the contract compares hashes
across collections. An earlier version of the contract used the bare hash as a
global key, which invited exactly the cross-molecule comparison the
representation does not support; that design is not reused here.

Two identifiers are stored per conformer:

- `cshHash` — the 256-bit soft hash, filling one storage slot exactly.
- `sha256Id` — the full SHA-256 of the canonical feature vector, used as the
  exact identifier and for deduplication. The soft hash is lossy and collisions
  are expected, so it is deliberately not used for that.

Measured cost on a local EVM, optimiser at 200 runs: deployment about 1.02 M
gas, opening a collection about 185 k, a single conformer about 128 k, and
about 98 k per conformer in batches of ten.

Requires `ConformerRegistry.abi.json` and `ConformerRegistry.bin` next to the
notebook (produced by compiling `ConformerRegistry.sol` with solc 0.8.20), plus
`SEPOLIA_RPC` and `SEPOLIA_PRIVATE_KEY` in the environment. Never put a key in
the notebook.

In [ ]:
# ── Connect to Sepolia ───────────────────────────────────────────────────────
from web3 import Web3

SEPOLIA_RPC = os.environ.get("SEPOLIA_RPC", "")
SEPOLIA_KEY = os.environ.get("SEPOLIA_PRIVATE_KEY", "")

# Leave empty to deploy a fresh registry; set it to reuse one already deployed.
CONTRACT_ADDRESS = "0x42fEcFe9B797e83300Bf10F39557e8d7f9B9FED0"

REGISTER    = False       # irreversible once True
BATCH_SIZE  = 10          # conformers per transaction
N_CONFS_REG = 20          # conformers per molecule to register

# Molecules chosen to span the range of outcomes rather than to show the method
# at its best: one where it does well, one where a baseline does better, one
# where it does worst in the whole study, and one rigid molecule where the
# baselines report no difference at all.
REGISTER_MOLECULES = ["Ketamine", "Imatinib", "Atorvastatin", "Caffeine"]

try:
    with open("ConformerRegistry.abi.json") as fh:
        CONTRACT_ABI = json.load(fh)
    with open("ConformerRegistry.bin") as fh:
        CONTRACT_BIN = fh.read().strip()
    artifacts = True
except FileNotFoundError:
    artifacts = False
    print("ConformerRegistry.abi.json / .bin not found — compile the contract first:")
    print("  solc --optimize --optimize-runs 200 --combined-json abi,bin ConformerRegistry.sol")

w3 = None
if SEPOLIA_RPC and artifacts:
    w3 = Web3(Web3.HTTPProvider(SEPOLIA_RPC))
    if w3.is_connected():
        acct = w3.eth.account.from_key(SEPOLIA_KEY).address if SEPOLIA_KEY else None
        bal = w3.from_wei(w3.eth.get_balance(acct), "ether") if acct else 0
        print(f"connected to chain {w3.eth.chain_id} | account {acct} | {bal:.4f} ETH")
    else:
        print("RPC unreachable")
        w3 = None
else:
    print("SEPOLIA_RPC not set — registration section will not run")

In [ ]:
# ── Deploy and register ──────────────────────────────────────────────────────
# Nonces are tracked locally rather than re-queried before every transaction:
# a node may not yet reflect a transaction it has already accepted, which makes
# get_transaction_count return a stale value and the next send fail with
# "nonce too low". The counter is seeded once from the pending block.
_nonce = {"next": None}

TX_TIMEOUT   = 300     # seconds to wait for a receipt
PRIORITY_GWEI = 2      # tip to the proposer; testnet base fees are often ~0,
                       # and a transaction priced off base fee alone can sit
                       # in the mempool indefinitely

def _fees(w3):
    """EIP-1559 fees with a real tip, so transactions actually get included."""
    tip = w3.to_wei(PRIORITY_GWEI, "gwei")
    try:
        base = w3.eth.get_block("latest")["baseFeePerGas"]
        return {"maxPriorityFeePerGas": tip, "maxFeePerGas": base * 2 + tip}
    except KeyError:                      # pre-1559 chain or a test provider
        return {"gasPrice": max(w3.eth.gas_price, tip)}


def _send(w3, fn, account, key, gas=None):
    """Sign and send one transaction, returning its receipt."""
    if _nonce["next"] is None:
        _nonce["next"] = w3.eth.get_transaction_count(account, "pending")
    for attempt in range(3):
        params = {
            "from":    account,
            "nonce":   _nonce["next"],
            "gas":     gas or int(fn.estimate_gas({"from": account}) * 1.25),
            "chainId": w3.eth.chain_id,
        }
        fees = _fees(w3)
        # each retry raises the tip, in case the first price was too low
        if "maxPriorityFeePerGas" in fees and attempt:
            fees["maxPriorityFeePerGas"] *= (attempt + 1)
            fees["maxFeePerGas"] += fees["maxPriorityFeePerGas"]
        params.update(fees)

        tx = fn.build_transaction(params)
        signed = w3.eth.account.sign_transaction(tx, key)
        raw = getattr(signed, "raw_transaction", None) or signed.rawTransaction
        try:
            h = w3.eth.send_raw_transaction(raw)
        except Exception as exc:
            msg = str(exc).lower()
            if ("nonce" in msg or "already known" in msg) and attempt < 2:
                _nonce["next"] = w3.eth.get_transaction_count(account, "pending")
                continue
            raise

        try:
            r = w3.eth.wait_for_transaction_receipt(h, timeout=TX_TIMEOUT)
            _nonce["next"] += 1
            return r
        except Exception:
            # Not mined in time. The transaction may still land later, so the
            # nonce is left alone and resynced; re-running this cell skips
            # whatever did make it on chain.
            print(f"    tx {h.hex()} not mined in {TX_TIMEOUT}s — retrying with a higher tip")
            _nonce["next"] = w3.eth.get_transaction_count(account, "pending")
            continue
    raise RuntimeError("could not get a transaction mined after 3 attempts; "
                       "check the account on sepolia.etherscan.io before re-running")


def deploy_registry(w3, account, key):
    c = w3.eth.contract(abi=CONTRACT_ABI, bytecode=CONTRACT_BIN)
    r = _send(w3, c.constructor(), account, key)
    print(f"deployed at {r.contractAddress} | gas {r.gasUsed:,}")
    return r.contractAddress


def _registered_ids(registry, collection_id):
    """sha256 identifiers already stored in a collection, so a re-run resumes
    instead of reverting on the duplicate check."""
    out = set()
    for i in range(registry.functions.conformerCount(collection_id).call()):
        out.add(registry.functions.getConformer(collection_id, i).call()[1].hex())
    return out


def register_molecules(w3, registry, account, key, molecules, n_confs=N_CONFS_REG):
    """Register every conformer of each molecule into its own collection."""
    records, total_gas = [], 0

    for name, smiles, _ in molecules:
        mol = embed(smiles, n_confs)
        nc = mol.GetNumConformers()
        H = CSHHasher(smiles, heavy=HEAVY, pca_dims=PCA_DIMS, n_bits=N_BITS)
        spec = H.spec_version()

        found, cid = registry.functions.findCollection(name, spec).call()
        if not found:
            r = _send(w3, registry.functions.openCollection(name, spec), account, key)
            total_gas += r.gasUsed
            _, cid = registry.functions.findCollection(name, spec).call()
            print(f"{name}: collection {cid} opened ({r.gasUsed:,} gas)")
        else:
            print(f"{name}: collection {cid} already open")

        # energies are metadata on the record; they are not part of the hash
        props = AllChem.MMFFGetMoleculeProperties(mol)
        hashes, ids, energies = [], [], []
        for c in range(nc):
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=c)
            hashes.append(bytes.fromhex(H.hash_hex(mol, c)))
            ids.append(bytes.fromhex(H.sha256_id(mol, c)))
            energies.append(int(ff.CalcEnergy() * 1000))

        already = _registered_ids(registry, cid)
        todo = [i for i in range(nc) if ids[i].hex() not in already]
        if len(todo) < nc:
            print(f"  {nc - len(todo)} conformer(s) already on chain, skipping those")

        for start in range(0, len(todo), BATCH_SIZE):
            idx = todo[start:start + BATCH_SIZE]
            r = _send(w3, registry.functions.registerBatch(
                cid, [hashes[i] for i in idx], [ids[i] for i in idx],
                [energies[i] for i in idx]), account, key)
            total_gas += r.gasUsed
            print(f"  conformers {idx[0]}-{idx[-1]}: {r.gasUsed:,} gas, "
                  f"block {r.blockNumber}")

        for c in range(nc):
            records.append({"molecule": name, "collection_id": cid,
                            "spec_version": spec, "conf_id": c,
                            "csh_hash": hashes[c].hex(),
                            "sha256_id": ids[c].hex(),
                            "energy_kcal": round(energies[c] / 1000, 3)})

    print(f"\ntotal gas this run: {total_gas:,}")
    return pd.DataFrame(records)


if REGISTER and w3 is not None and SEPOLIA_KEY:
    account = w3.eth.account.from_key(SEPOLIA_KEY).address
    address = CONTRACT_ADDRESS or deploy_registry(w3, account, SEPOLIA_KEY)
    registry = w3.eth.contract(address=Web3.to_checksum_address(address),
                               abi=CONTRACT_ABI)
    chosen = [m for m in DEV + CONTROL if m[0] in REGISTER_MOLECULES]
    df_chain = register_molecules(w3, registry, account, SEPOLIA_KEY, chosen)
    df_chain.to_csv("registry_records.csv", index=False)
    print(f"saved -> registry_records.csv ({len(df_chain)} records)")
    print(f"contract: {address}")
else:
    print("REGISTER is False — nothing was written to the blockchain.")
    if artifacts:
        print("Set REGISTER = True, SEPOLIA_RPC and SEPOLIA_PRIVATE_KEY to register.")

## 12. Build the demo data files

Reads the `ConformerRegistered` events back from the chain and writes the two
JSON files the demo site serves. Everything here comes from the chain rather
than from local state, so the site shows what was actually registered.

Benchmark values are attached per molecule so the site can state plainly where
the method does well and where it does not, instead of showing only its best
case.

In [ ]:
# ── Build conformers.json and stats.json from on-chain events ────────────────
DEMO_DIR   = "demo_data"
LOG_WINDOW = 10        # Alchemy's free tier caps eth_getLogs at a 10-block range
RPC_PAUSE  = 0.06      # seconds between calls, to stay under the rate limit

# Spearman rho against DM-RMSD, from the benchmark above. Included so the demo
# can report each molecule honestly rather than showing only favourable cases.
BENCH = {
    "Caffeine":     {"dmRmsdMax": 0.000, "csh": 0.377, "best": None,  "bestName": None},
    "Imatinib":     {"dmRmsdMax": 3.507, "csh": 0.691, "best": 0.928, "bestName": "3DAPfp"},
    "Atorvastatin": {"dmRmsdMax": 2.033, "csh": 0.399, "best": 0.762, "bestName": "USRCAT"},
    "Ketamine":     {"dmRmsdMax": 0.806, "csh": 0.890, "best": 0.677, "bestName": "USRCAT"},
}

def _rpc(fn, *args, tries=5, **kwargs):
    """Call an RPC-backed function, backing off on rate limits."""
    for i in range(tries):
        try:
            time.sleep(RPC_PAUSE)
            return fn(*args, **kwargs)
        except Exception as exc:
            if "429" in str(exc) and i < tries - 1:
                time.sleep(2 ** i)
                continue
            raise


def deployment_block(w3, address, latest=None):
    """Block in which the contract was created, by binary search on eth_getCode.
    About 25 calls for the whole chain, and it keeps the event scan down to the
    blocks that can actually contain records."""
    addr = Web3.to_checksum_address(address)
    lo, hi = 0, latest if latest is not None else w3.eth.block_number
    if _rpc(w3.eth.get_code, addr, block_identifier=hi) in (b"", "0x"):
        raise ValueError(f"no contract at {address}")
    while lo < hi:
        mid = (lo + hi) // 2
        if _rpc(w3.eth.get_code, addr, block_identifier=mid) in (b"", "0x"):
            lo = mid + 1
        else:
            hi = mid
    return lo


def _bits(hex_str):
    return np.unpackbits(np.frombuffer(bytes.fromhex(hex_str), dtype=np.uint8))


def build_demo_data(w3, registry, address, out_dir=DEMO_DIR):
    """Rebuild the demo data from chain state.

    Everything a record needs is already in the ConformerRegistered event, so
    the events are the only per-conformer source used here: reading each record
    back individually would multiply the request count by the number of
    conformers for no extra information.
    """
    os.makedirs(out_dir, exist_ok=True)

    n_coll = _rpc(registry.functions.collectionCount().call)
    meta = {}
    expected = 0
    for cid in range(n_coll):
        molecule, spec, owner, created, count = _rpc(
            registry.functions.getCollection(cid).call)
        meta[cid] = {"molecule": molecule, "spec": spec, "owner": owner,
                     "createdAt": created, "count": count}
        expected += count
    print(f"{n_coll} collections, {expected} conformers registered")

    latest = _rpc(lambda: w3.eth.block_number)
    start = deployment_block(w3, address, latest)
    print(f"scanning from block {start} in windows of {LOG_WINDOW} "
          f"(stops once all {expected} events are found)")

    events, lo = {}, start
    while lo <= latest and len(events) < expected:
        hi = min(lo + LOG_WINDOW - 1, latest)
        try:
            for e in _rpc(registry.events.ConformerRegistered().get_logs,
                          from_block=lo, to_block=hi):
                events[(e["args"]["collectionId"], e["args"]["index"])] = e
        except Exception as exc:
            print(f"  blocks {lo}-{hi}: {type(exc).__name__}")
        lo = hi + 1
    print(f"{len(events)} of {expected} events recovered")

    # block timestamps, fetched once per block rather than once per conformer
    blocks = sorted({e["blockNumber"] for e in events.values()})
    times = {}
    for b in blocks:
        try:
            times[b] = _rpc(w3.eth.get_block, b)["timestamp"]
        except Exception:
            times[b] = None

    conformers, collections, spec_seen = [], [], set()
    for cid in range(n_coll):
        m = meta[cid]
        spec_seen.add(m["spec"])
        hashes = []
        for i in range(m["count"]):
            e = events.get((cid, i))
            if e is None:
                continue
            a = e["args"]
            h = a["cshHash"].hex().removeprefix("0x")
            hashes.append(h)
            conformers.append({
                "id":           f"{m['molecule'].lower()}_conf{i:02d}",
                "molecule":     m["molecule"],
                "collectionId": cid,
                "confId":       i,
                "hash":         h,
                "sha256":       a["sha256Id"].hex().removeprefix("0x"),
                "energy":       a["energyMilli"] / 1000.0,
                "block":        e["blockNumber"],
                "tx":           "0x" + e["transactionHash"].hex().removeprefix("0x"),
                "timestamp":    times.get(e["blockNumber"]),
            })

        # Hamming distances are computed only within a collection. Distances
        # between collections are not defined for this representation and are
        # deliberately never computed anywhere in this file.
        bits = [_bits(h) for h in hashes]
        d = [float(np.mean(bits[i] != bits[j]))
             for i in range(len(bits)) for j in range(i + 1, len(bits))]
        b = BENCH.get(m["molecule"], {})
        collections.append({
            "id": cid, "molecule": m["molecule"], "specVersion": m["spec"],
            "owner": m["owner"], "createdAt": m["createdAt"], "n": len(hashes),
            "uniqueHashes": len(set(hashes)),
            "meanHamming": round(float(np.mean(d)), 4) if d else None,
            "minHamming":  round(float(np.min(d)), 4) if d else None,
            "maxHamming":  round(float(np.max(d)), 4) if d else None,
            "dmRmsdMax":   b.get("dmRmsdMax"),
            "rhoCSH":      b.get("csh"),
            "rhoBest":     b.get("best"),
            "bestMethod":  b.get("bestName"),
        })
        print(f"  collection {cid}: {m['molecule']}, {len(hashes)} conformers, "
              f"{len(set(hashes))} unique hashes")

    stats = {
        "contract":          address,
        "chain":             "sepolia",
        "chainId":           w3.eth.chain_id,
        "hashBits":          N_BITS,
        "conformersOnChain": len(conformers),
        "collectionCount":   n_coll,
        "lastIndexedBlock":  max(blocks, default=0),
        "specVersion":       sorted(spec_seen)[0] if len(spec_seen) == 1 else sorted(spec_seen),
        "collections":       collections,
        # Kept for the record; these ran an earlier, superseded pipeline and
        # their hashes are not comparable with the ones above.
        "legacyContracts": ["0x376E4dB2943Cd63e8d0C6B8278d8fb378F28c9fa",
                            "0x2081c6b3216Bb21811c179C195cbe9490C9F6572"],
    }

    with open(f"{out_dir}/conformers.json", "w") as fh:
        json.dump(conformers, fh, indent=1)
    with open(f"{out_dir}/stats.json", "w") as fh:
        json.dump(stats, fh, indent=1)
    print(f"\nwrote {out_dir}/conformers.json ({len(conformers)} records)")
    print(f"wrote {out_dir}/stats.json")
    return conformers, stats


if w3 is not None and CONTRACT_ADDRESS:
    _reg = w3.eth.contract(address=Web3.to_checksum_address(CONTRACT_ADDRESS),
                           abi=CONTRACT_ABI)
    demo_conformers, demo_stats = build_demo_data(w3, _reg, CONTRACT_ADDRESS)
else:
    print("Set CONTRACT_ADDRESS and connect to a node to build the demo data.")